# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [2]:
# Desarrollo de la actividad 1
import pandas as pd
ds_eva = pd.read_csv('../kit/data/raw/eva.csv')
ds_eva.info()
print('----')
# Preparar las cuatro columnas de trabajo en eva.
#codigo_departamento= ds_eva[['Código Dane departamento','Año','Área cosechada','Producción']]
ds_eva["codigo_departamento"] = ds_eva["Código Dane departamento"]
ds_eva["anio"] = pd.to_numeric(ds_eva["Año"], errors="coerce").astype("Int64")
ds_eva["area_ha"] = pd.to_numeric(ds_eva["Área cosechada"], errors="coerce")
ds_eva["produccion_t"] = pd.to_numeric(ds_eva["Producción"], errors="coerce")
display(ds_eva[["codigo_departamento","anio","area_ha","produccion_t"]])
# Mostrar los tipos y los valores que no pudieron convertirse.
import pandas as pd
cod = ['codigo_departamento','anio','area_ha','produccion_t']
registros = []

for campo in cod:
    s = ds_eva[campo]
    
    # Nulos nativos de pandas (NaN / None)
    nulos = s.isna().sum()
    
    # Cadenas no nulas con espacios removidos
    s_str = s.dropna().astype(str).str.strip()
    
    # Textos vacíos tras retirar espacios
    textos_vacios = s_str.eq("").sum()
    
    # Marcadores "ND" 
    marcadores_nd = s_str.str.upper().eq("ND").sum()
    
    registros.append({
        "campo": campo,
        "nulos": nulos,
        "textos_vacios": textos_vacios,
        "marcadores_ND": marcadores_nd
    })

# 2. Construir el DataFrame 'perfil' final
codigo_departamento = pd.DataFrame(registros)
print("-------------------")
display(codigo_departamento)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166732 entries, 0 to 166731
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Código Dane departamento       166732 non-null  int64  
 1   Departamento                   166732 non-null  object 
 2   Código Dane municipio          166732 non-null  int64  
 3   Municipio                      166732 non-null  object 
 4   Grupo cultivo                  166732 non-null  object 
 5   Subgrupo                       166732 non-null  object 
 6   Cultivo                        166732 non-null  object 
 7   Desagregación cultivo          166732 non-null  object 
 8   Año                            166732 non-null  int64  
 9   Periodo                        166732 non-null  object 
 10  Área sembrada                  166732 non-null  float64
 11  Área cosechada                 166732 non-null  float64
 12  Producción                    

,codigo_departamento,anio,area_ha,produccion_t
0,5,2021,8.0,128.00
1,5,2021,8.0,152.00
2,5,2022,8.0,127.44
3,5,2022,8.0,152.00
4,5,2023,7.0,133.00
...,...,...,...,...
166727,99,2021,380.0,3420.00
166728,99,2022,380.0,3800.00
166729,99,2023,350.0,3500.00
166730,99,2024,400.0,4400.00


-------------------


,campo,nulos,textos_vacios,marcadores_ND
0,codigo_departamento,0,0,0
1,anio,0,0,0
2,area_ha,0,0,0
3,produccion_t,0,0,0


**Interpretación del resultado:**

_Se inspecciono la tabla "eva", donde se pudieron ver los datos iniciales de la misma para cada columna y como estan nombradas las mismas.Posteriormente, se creo hicieron la  creacion de las columnas solicitadas como se ve anteriormente.Tambien se realizo una tabla con las posibles ausencias y no se encuentra ninguna._

_Por otro lado, se comprobo que hay 166732 registros._

## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [3]:
# Desarrollo de la actividad 2
# Construir la máscara y guardar consulta_loc.
# Mostrar los conteos de entrada, selección y diferencia.
consulta_loc = ds_eva.loc[
    #ds_eva["codigo_departamento"].eq(15)
    ds_eva["Departamento"].eq("Boyacá")
    & ds_eva["anio"].isin([2023, 2024])
    & ds_eva["area_ha"].gt(0)
    & ds_eva["produccion_t"].ge(0)
].copy()

total_original = len(ds_eva)
total_seleccionado = len(consulta_loc)
diferencia = total_original - total_seleccionado

print("Total original:", total_original)
print("Total seleccionado:", total_seleccionado)
print("Diferencia:", diferencia)
print("-------------------")
display(consulta_loc)

Total original: 166732
Total seleccionado: 4924
Diferencia: 161808
-------------------


,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,...,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo,codigo_departamento,anio,area_ha,produccion_t
22910,15,Boyacá,15001,Tunja,Raíces y tubérculos,Raíces y tubérculos,Arracacha,Arracacha,2023,2023A,...,3750.00,25.00,Transitorio,En fresco,1080200,Arracacia xanthorrhiza,15,2023,150.0,3750.00
22912,15,Boyacá,15001,Tunja,Raíces y tubérculos,Raíces y tubérculos,Arracacha,Arracacha,2024,2024A,...,1250.00,25.00,Transitorio,En fresco,1080200,Arracacia xanthorrhiza,15,2024,50.0,1250.00
22913,15,Boyacá,15001,Tunja,Raíces y tubérculos,Raíces y tubérculos,Arracacha,Arracacha,2024,2024B,...,100.00,25.00,Transitorio,En fresco,1080200,Arracacia xanthorrhiza,15,2024,4.0,100.00
22924,15,Boyacá,15001,Tunja,Leguminosas,Leguminosas,Arveja,Arveja,2023,2023A,...,29.44,1.64,Transitorio,Grano,1060100,Pisum sativum,15,2023,18.0,29.44
22925,15,Boyacá,15001,Tunja,Leguminosas,Leguminosas,Arveja,Arveja,2023,2023B,...,747.66,1.87,Transitorio,Grano,1060100,Pisum sativum,15,2023,400.0,747.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40567,15,Boyacá,15897,Zetaquira,Frutales,Demás frutales,Plátano,Plátano consumo interno,2024,2024,...,242.40,8.00,Permanente,En fresco,2045802,Musa x paradisiaca,15,2024,30.3,242.40
40578,15,Boyacá,15897,Zetaquira,Hortalizas,Hortalizas de fruto,Tomate,Tomate invernadero,2023,2023A,...,270.00,45.00,Transitorio,En fresco,1052902,Lycopersicon esculentum,15,2023,6.0,270.00
40579,15,Boyacá,15897,Zetaquira,Hortalizas,Hortalizas de fruto,Tomate,Tomate invernadero,2023,2023B,...,270.00,45.00,Transitorio,En fresco,1052902,Lycopersicon esculentum,15,2023,6.0,270.00
40580,15,Boyacá,15897,Zetaquira,Hortalizas,Hortalizas de fruto,Tomate,Tomate invernadero,2024,2024A,...,360.00,45.00,Transitorio,En fresco,1052902,Lycopersicon esculentum,15,2024,8.0,360.00


**Interpretación del resultado:**

_Se selecciono el codigo del departamento de Boyacá o el nombre del mismo.Además, de los datos de 2023 a 2024 de área cosechada positiva y producción no negativa.Lo anterior, se guardo en "consulta_loc".Anteriormente se ve el valor inicial u original, los valores seleccionados del mismo y la diferencia entre los dos totales, lo cual se puede deber a que hay registros fuera del alcance territorial o temporal, y no por eso implica que todos sean errores._

## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [4]:
# Desarrollo de la actividad 3
# Construir consulta_query.
# Comprobar que consulta_loc y consulta_query son equivalentes.
departamento = "Boyacá"

consulta_query = ds_eva.query(
    "Departamento == @departamento and "
    "anio in [2023, 2024] and "
    "area_ha > 0 and "
    "produccion_t >= 0",
    engine="python"
).copy()
display(consulta_query)
pd.testing.assert_frame_equal(
    consulta_loc,
    consulta_query
)


,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,...,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo,codigo_departamento,anio,area_ha,produccion_t
22910,15,Boyacá,15001,Tunja,Raíces y tubérculos,Raíces y tubérculos,Arracacha,Arracacha,2023,2023A,...,3750.00,25.00,Transitorio,En fresco,1080200,Arracacia xanthorrhiza,15,2023,150.0,3750.00
22912,15,Boyacá,15001,Tunja,Raíces y tubérculos,Raíces y tubérculos,Arracacha,Arracacha,2024,2024A,...,1250.00,25.00,Transitorio,En fresco,1080200,Arracacia xanthorrhiza,15,2024,50.0,1250.00
22913,15,Boyacá,15001,Tunja,Raíces y tubérculos,Raíces y tubérculos,Arracacha,Arracacha,2024,2024B,...,100.00,25.00,Transitorio,En fresco,1080200,Arracacia xanthorrhiza,15,2024,4.0,100.00
22924,15,Boyacá,15001,Tunja,Leguminosas,Leguminosas,Arveja,Arveja,2023,2023A,...,29.44,1.64,Transitorio,Grano,1060100,Pisum sativum,15,2023,18.0,29.44
22925,15,Boyacá,15001,Tunja,Leguminosas,Leguminosas,Arveja,Arveja,2023,2023B,...,747.66,1.87,Transitorio,Grano,1060100,Pisum sativum,15,2023,400.0,747.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40567,15,Boyacá,15897,Zetaquira,Frutales,Demás frutales,Plátano,Plátano consumo interno,2024,2024,...,242.40,8.00,Permanente,En fresco,2045802,Musa x paradisiaca,15,2024,30.3,242.40
40578,15,Boyacá,15897,Zetaquira,Hortalizas,Hortalizas de fruto,Tomate,Tomate invernadero,2023,2023A,...,270.00,45.00,Transitorio,En fresco,1052902,Lycopersicon esculentum,15,2023,6.0,270.00
40579,15,Boyacá,15897,Zetaquira,Hortalizas,Hortalizas de fruto,Tomate,Tomate invernadero,2023,2023B,...,270.00,45.00,Transitorio,En fresco,1052902,Lycopersicon esculentum,15,2023,6.0,270.00
40580,15,Boyacá,15897,Zetaquira,Hortalizas,Hortalizas de fruto,Tomate,Tomate invernadero,2024,2024A,...,360.00,45.00,Transitorio,En fresco,1052902,Lycopersicon esculentum,15,2024,8.0,360.00


**Interpretación del resultado:**

_Se repitio la misma seleccion del punto anterior mediante .query, donde el resultado se guardo en "consulta_query".Posteriormente, se compararon "consulta_loc" y "consulta_query", y se comprobo que tienen la mismas filas, columnas y valores. Tambien como en el caso anterior, solo se seleccionaron 4924 registros._

## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [5]:
# Desarrollo de la actividad 4
# Crear resumen con registros, total_anio y porcentaje_anual.
resumen = (
    consulta_loc
    .groupby(["anio", "Cultivo"], dropna=False)
    .size()
    .reset_index(name="registros")
)

resumen["total_anual"] = (
    resumen.groupby("anio")["registros"]
    .transform("sum")
)

resumen["porcentaje_anual"] = (
    resumen["registros"] / resumen["total_anual"] * 100
)

resumen = resumen.sort_values(
    ["anio", "registros"],
    ascending=[True, False]
).reset_index(drop=True)
# Verificar los totales y los porcentajes por año.
print("Registros en consulta_loc:", len(consulta_loc))
print("Registros en resumen:", resumen["registros"].sum())

comprobacion = (
    resumen
    .groupby("anio")["porcentaje_anual"]
    .sum()
)
print("-----------")
display(comprobacion)
print("------------")
display(
    resumen.assign(
        porcentaje_anual=resumen["porcentaje_anual"].round(2)
    )
)

Registros en consulta_loc: 4924
Registros en resumen: 4924
-----------


anio
2023    100.0
2024    100.0
Name: porcentaje_anual, dtype: float64

------------


,anio,Cultivo,registros,total_anual,porcentaje_anual
0,2023,Papa,239,2436,9.81
1,2023,Maíz,194,2436,7.96
2,2023,Arveja,176,2436,7.22
3,2023,Frijol,162,2436,6.65
4,2023,Tomate,113,2436,4.64
...,...,...,...,...,...
187,2024,Dátil,1,2488,0.04
188,2024,Macadamia,1,2488,0.04
189,2024,Olivo,1,2488,0.04
190,2024,"Otros cultivos para condimentos, bebidas medic...",1,2488,0.04


**Interpretación del resultado:**

_Se construyo "resumen" sobre consulta_loc, se crearon las columnas adiccionales solicitadas y se ordenaron los resultados por año y cantidad de resgistros de forma descendente.Por otro lado, la suma de los registros si coinciden con el tamaño de consulta_loc y los porcentajes suman aproximadamente 100% por año._

## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [6]:
# Desarrollo de la actividad 5
# Construir y mostrar la tabla dinámica.
# Exportar resumen y explicar dos resultados en la celda siguiente.
from pathlib import Path
OUT = Path("OUT")
OUT.mkdir(exist_ok=True)
tabla_dinamica = resumen.pivot_table(
    index="Cultivo",
    columns="anio",
    values="registros",
    aggfunc="sum",
    fill_value=0
)

print(tabla_dinamica)

resumen.to_csv(OUT / "resumen_eva.csv", index=False)

print("Total resumen:", resumen["registros"].sum())
print("Total tabla dinámica:", tabla_dinamica.to_numpy().sum())
print(OUT / "resumen_eva.csv")

anio             2023  2024
Cultivo                    
Acelga              5     8
Achira              4     4
Agraz - mortiño     3     4
Aguacate           48    52
Ahuyama            23    26
...               ...   ...
Uchuva             25    21
Uva                 6     6
Yuca               82    82
Zanahoria          44    46
Zapote              5     6

[97 rows x 2 columns]
Total resumen: 4924
Total tabla dinámica: 4924
OUT/resumen_eva.csv


**Interpretación del resultado:**

_Se contruyo la tabla dinamica a partir de resumen, con cultivos en filas, años en columnas y conteos como valores. además, se completarons las combinaciones sin registros, para luego exportar resumen a OUT / "resumen_eva.csv" y verificar que se halla creado._

Se pudo observar que el total de registros entregados por resumen son los mismos que se contabilizan o cuentan del total de la tabla dinamica.

En la tabla dinamica, se puede ver que baja el numero de filas, pero esto se debe a que se agrupan los registros de un cultivo en una sola fila, pero siguen existiendo los 4924 registros que entrego la tabla "resumen".

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.